# 드럼 분리 벤치마크 — 코랩 실행

저장소 `AI/` 코드와 구글 드라이브의 데이터로 `ssaplash-bench` 명령을 그대로 돌린다 (설계 4-3절, ADR-008·011).
셀마다 명령만 부르고 로직은 없다. 결과·MLflow 기록은 드라이브에 바로 써서 세션이 끊겨도 남는다.

준비물 (드라이브 `MyDrive/ssaplash/`)
- `S15P31A307.zip`: 저장소를 zip으로 묶은 것 (`git archive`, 데이터 없음. 미결 U7: 1차는 zip 업로드)
- `data/musdb18hq/test/`: MUSDB18-HQ 테스트 50곡 (11GB). train은 올리지 않는다
- `data/synth_v1/`: 합성곡 5곡 (216MB). GMD는 올리지 않는다

런타임은 GPU(코랩 프로+ A100 또는 L4)로 바꾼 뒤 위에서부터 실행한다. GPU는 한 종류로만 측정한다.

**세션이 끊기면** 런타임을 다시 연결하고 1~4를 다시 실행한 뒤 6·7 셀을 그대로 다시 실행한다.
runId가 고정돼 있어 이미 끝난 곡은 건너뛰고 실패한 곡만 다시 돈다(이어하기). 3번 셀이 찍는 GPU 이름이
이전과 다르면 결과 CSV의 `gpuName` 열이 섞이므로, 런타임을 바꿔 같은 GPU를 받은 뒤 이어 간다.


In [ ]:
# 1. 드라이브 연결
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ssaplash"
!ls -la $DRIVE $DRIVE/data

In [ ]:
# 2. 저장소 풀기 (zip 방식). 이미 풀려 있으면 건너뜀. 코드를 새로 올렸으면 /content/S15P31A307 을 지우고 다시 실행
import os
if not os.path.isdir("/content/S15P31A307/AI"):
    !unzip -q "$DRIVE/S15P31A307.zip" -d /content/
%cd /content/S15P31A307/AI
!git log --oneline -1 2>/dev/null || echo "git 정보 없음 (zip)"

In [ ]:
# 3. uv 설치 + 의존성 (모델 실행 extra 포함). 한글 그래프용 글꼴도 설치. 마지막 줄의 GPU 이름을 기록해 둔다
# 코랩 커널의 MPLBACKEND(matplotlib_inline)는 uv 가상환경에 없는 모듈이라 Agg로 덮어씀 (cli도 같은 처리)
%env MPLBACKEND=Agg
!pip install -q uv
!apt-get install -qq -y fonts-nanum > /dev/null
!uv sync --extra models
!uv run python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

In [ ]:
# 4. 데이터를 런타임 로컬 디스크로 복사 (드라이브 직접 읽기는 느리고 긴 실행 중 끊김). 이미 있으면 건너뜀
!mkdir -p data/musdb18hq
![ -d data/musdb18hq/test ] || cp -r "$DRIVE/data/musdb18hq/test" data/musdb18hq/
![ -d data/synth_v1 ] || cp -r "$DRIVE/data/synth_v1" data/
!du -sh data/musdb18hq/test data/synth_v1
!uv run ssaplash-bench check-data configs/bench/datasets/musdb18hq_test.yaml configs/bench/datasets/synth_v1.yaml

In [ ]:
# 5. 동작 확인: 모델 1개 × 30초 클립 3곡. 결과·MLflow가 드라이브에 써지는지까지 본다
!PYTHONUTF8=1 uv run ssaplash-bench run \
    --models configs/bench/models/msst/scnet_large.yaml \
    --dataset configs/bench/datasets/musdb18hq_test_clip30.yaml \
    --out "$DRIVE/results" --run-id clip30-check --mlflow-uri "$DRIVE/mlruns"

In [ ]:
# 6. 본측정 1: 모델 전부 × MUSDB 테스트 50곡 (수 시간). runId 고정 → 끊기면 이 셀을 다시 실행하면 이어서 돈다
MODELS = " ".join(f"configs/bench/models/{m}.yaml" for m in [
    "msst/scnet_xl_ihf", "msst/bs_roformer_sw", "msst/scnet_large", "msst/bs_roformer_musdb",
    "demucs/htdemucs_ft", "demucs/htdemucs", "openunmix/umxhq",
])
!PYTHONUTF8=1 uv run ssaplash-bench run --models $MODELS \
    --dataset configs/bench/datasets/musdb18hq_test.yaml \
    --out "$DRIVE/results" --run-id musdb-full --mlflow-uri "$DRIVE/mlruns" --save-wav 3

In [ ]:
# 7. 본측정 2: 합성곡 5곡 (정답 온셋이 MIDI). 끊기면 다시 실행
MODELS = " ".join(f"configs/bench/models/{m}.yaml" for m in [
    "msst/scnet_xl_ihf", "msst/bs_roformer_sw", "msst/scnet_large", "msst/bs_roformer_musdb",
    "demucs/htdemucs_ft", "demucs/htdemucs", "openunmix/umxhq",
])
!PYTHONUTF8=1 uv run ssaplash-bench run --models $MODELS \
    --dataset configs/bench/datasets/synth_v1.yaml \
    --out "$DRIVE/results" --run-id synth-full --mlflow-uri "$DRIVE/mlruns" --save-wav 5

In [ ]:
# 8. 두 실행을 합친 보고서 → 드라이브 results/combined/
!PYTHONUTF8=1 uv run ssaplash-bench report \
    "$DRIVE/results/musdb-full/results.csv" "$DRIVE/results/synth-full/results.csv" \
    --out "$DRIVE/results/combined"
!cat "$DRIVE/results/combined/summary.md"

로컬에서 보고서를 다시 만들려면 드라이브의 `results/<runId>/results.csv`를 받아 `uv run ssaplash-bench report <csv>`를 돌린다.
MLflow 기록은 `mlruns/` 폴더째 받아 `uv run mlflow ui --backend-store-uri mlruns`로 본다.
결과 CSV의 `gpuName` 열이 한 종류인지 확인한 뒤 문서에 쓴다.
